In [1]:
from pathlib import Path

import pandas as pd

TIMING_RESULTS = Path("..") / "timing_results_tsp"
DATA = Path("..") / "data"
UNIT_TO_S = {"MILLISECONDS": 1e-3, "SECONDS": 1.0, "NANOSECONDS": 1e-9, "MINUTES": 60.0}

# (method, cores) -> run directory under TIMING_RESULTS. HC2 has a 1-core run only.
RUNS = {
    ("MultiRocketHydra", 1): "mrhydra_1cpu/MultiRocketHydra",
    ("MultiRocketHydra", 16): "mrhydra_16cpu/MultiRocketHydra",
    ("HIVE-COTE 2.0", 1): "hc2_1cpu/HIVECOTEV2",
    ("TSCGlue", 1): "tscglue-high_1cpu/TSCGlueEnhancedV4-High-LogLoss-GPU",
    ("TSCGlue", 16): "tscglue-high_16cpu/TSCGlueEnhancedV4-High-LogLoss-GPU",
}


def n_instances(ts_file):
    lines = ts_file.read_text().splitlines()
    start = next(i for i, line in enumerate(lines) if line.strip().lower() == "@data") + 1
    return sum(1 for line in lines[start:] if line.strip())


rows = []
for (method, cores), run in RUNS.items():
    for f in sorted((TIMING_RESULTS / run).glob("Predictions/*/testResample0.csv")):
        with open(f) as fh:
            header = fh.readline().rstrip("\n").split(",")
            fh.readline()
            result = fh.readline().rstrip("\n").split(",")
        scale = UNIT_TO_S[header[4]]
        dataset = f.parent.name
        rows.append(
            {
                "method": method,
                "cores": cores,
                "dataset": dataset,
                "fit_s": float(result[1]) * scale,
                "predict_s": float(result[2]) * scale,
                "n_train": n_instances(DATA / dataset / f"{dataset}0_TRAIN.ts"),
                "n_test": n_instances(DATA / dataset / f"{dataset}0_TEST.ts"),
            }
        )

df = pd.DataFrame(rows)
df.groupby(["method", "cores"]).size()

method            cores
HIVE-COTE 2.0     1        112
MultiRocketHydra  1        112
                  16       112
TSCGlue           1        112
                  16       112
dtype: int64

In [2]:
metrics = {}
for method, cores in RUNS:
    g = df[(df["method"] == method) & (df["cores"] == cores)]
    fit, predict = g["fit_s"].sum(), g["predict_s"].sum()
    metrics[(method, cores)] = {
        "Total (min)": (fit + predict) / 60,
        "Train (min)": fit / 60,
        "Inference (min)": predict / 60,
        "Mean/dataset (min)": (fit + predict) / 60 / len(g),
        "Train inst./s": g["n_train"].sum() / fit,
        "Test inst./s": g["n_test"].sum() / predict,
    }

table = pd.DataFrame(metrics).round(2)
table.columns.names = ["Method", "Cores"]
table

Method             MultiRocketHydra         HIVE-COTE 2.0  TSCGlue         
Cores                            1       16            1        1        16
Total (min)                  107.89   15.62      13950.16  2159.54  1152.54
Train (min)                   52.07   10.17       7726.09  1294.29   454.15
Inference (min)               55.82    5.44       6224.07   865.25   698.39
Mean/dataset (min)             0.96    0.14        124.56    19.28    10.29
Train inst./s                 18.18   93.05          0.12     0.73     2.08
Test inst./s                  37.07  380.08          0.33     2.39     2.96